# LeetCode 849 — Maximize Distance to Closest Person bằng Python

Link đề: https://leetcode.com/problems/maximize-distance-to-closest-person/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Two Pointers, Binary Search (chặt nhị phân)

Skeleton LeetCode:

```python
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        ...
```

Ghi chú: bài rất ngắn nhưng chứa một bẫy kinh điển: **hai đầu hàng ghế** và **các khoảng trống ở giữa** tính khoảng cách khác nhau. Khoảng trống ở giữa cho `(độ rộng) // 2`; khoảng trống ở đầu/cuối cho **nguyên độ dài**. Notebook đi từ vét cạn tới quét hai chiều, một lượt, nhóm các số 0 và chặt nhị phân đáp án.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `seats[i] = 1` là có người, `0` là ghế trống; chọn một ghế trống sao cho **khoảng cách tới người gần nhất lớn nhất**; trả khoảng cách đó.
2. Nắm các công cụ Python cần thiết: `enumerate`, `max`/`min` với nhiều đối số, `itertools.groupby` để nhóm các số 0 liên tiếp, tổng tiền tố (`accumulate`), chặt nhị phân trên đáp án.
3. Hiểu cấu trúc lời giải: đáp án là `max` của **ba loại khoảng trống**: đầu (khoảng cách tới người đầu tiên), cuối, và giữa (`khoảng cách hai người // 2`).
4. Đi qua 5 cách: vét cạn từng ghế trống, hai lượt tính khoảng cách trái/phải, một lượt theo người đầu/cuối và khoảng trống lớn nhất, nhóm các số 0 bằng `groupby`, chặt nhị phân trên đáp án; đối chiếu bằng oracle.
5. Tránh các lỗi kinh điển: bỏ qua hai đầu, dùng `// 2` cho đầu/cuối, lệch 1 ở khoảng trống giữa, dùng `/` ra số thực, và không xử lý trường hợp "chưa gặp người nào".

## 2. Đề bài & ví dụ

Cho mảng `seats` biểu diễn một hàng ghế: `seats[i] = 1` nghĩa là có người ngồi ở ghế thứ `i`, `seats[i] = 0` nghĩa là ghế trống (đánh số từ 0).

Có **ít nhất một ghế trống** và **ít nhất một người** đang ngồi.

Alex muốn ngồi vào một ghế trống sao cho **khoảng cách từ anh đến người gần nhất** là **lớn nhất**. Trả về khoảng cách lớn nhất đó.

**Ví dụ 1**

- Input: `seats = [1,0,0,0,1,0,1]`
- Output: `2`
- Giải thích: ngồi ở ghế 2 thì người gần nhất cách 2 ghế; mọi ghế trống khác chỉ cách 1.

**Ví dụ 2**

- Input: `seats = [1,0,0,0]`
- Output: `3`
- Giải thích: ngồi ghế cuối (chỉ số 3), người gần nhất cách 3 ghế.

**Ví dụ 3**

- Input: `seats = [0,1]`
- Output: `1`

**Ràng buộc:**

- `2 <= seats.length <= 2 * 10^4`
- `seats[i]` là `0` hoặc `1`.
- Có ít nhất một ghế trống và ít nhất một ghế có người.

In [1]:
from typing import List
from itertools import groupby, accumulate
import random, time

examples = [
    ([1, 0, 0, 0, 1, 0, 1], 2),
    ([1, 0, 0, 0], 3),
    ([0, 1], 1),
]
for seats, expected in examples:
    print(f"seats={seats} kỳ vọng={expected}")

seats=[1, 0, 0, 0, 1, 0, 1] kỳ vọng=2
seats=[1, 0, 0, 0] kỳ vọng=3
seats=[0, 1] kỳ vọng=1


## 3. Kiến thức nền

### 3.1. Khoảng cách trên một hàng

Khoảng cách giữa ghế `i` và ghế `j` là `|i - j|`. Khoảng cách tới **người gần nhất** của ghế `i` là `min(|i - j|)` trên mọi `j` có người.

In [2]:
seats = [1, 0, 0, 0, 1, 0, 1]
people = [j for j, c in enumerate(seats) if c]
print("vị trí người:", people)
for i, c in enumerate(seats):
    if c == 0:
        print(f"ghế {i}: khoảng cách tới người gần nhất = {min(abs(i - j) for j in people)}")

vị trí người: [0, 4, 6]
ghế 1: khoảng cách tới người gần nhất = 1
ghế 2: khoảng cách tới người gần nhất = 2
ghế 3: khoảng cách tới người gần nhất = 1
ghế 5: khoảng cách tới người gần nhất = 1


### 3.2. Ba loại khoảng trống

Các ghế trống liên tiếp tạo thành **khoảng trống** (gap). Có ba loại, tùy hai bên của nó:

- **Đầu hàng**: các số 0 ở đầu mảng, bên phải là người đầu tiên. Ngồi **sát mép trái** (ghế 0): khoảng cách = số ghế từ ghế 0 tới người đầu tiên = **chỉ số của người đầu tiên**.
- **Cuối hàng**: tương tự, khoảng cách = `n - 1 - chỉ số của người cuối`.
- **Giữa hai người** ở `a < b`: ngồi chính giữa, khoảng cách = `(b - a) // 2` (làm tròn xuống vì ghế rời rạc).

In [3]:
seats = [0, 0, 1, 0, 0, 0, 1, 0]
people = [j for j, c in enumerate(seats) if c]
n = len(seats)
print("người ở", people, "n =", n)
print("đầu hàng:", people[0], "| cuối hàng:", n - 1 - people[-1])
print("giữa:", [(b - a) // 2 for a, b in zip(people, people[1:])])

người ở [2, 6] n = 8
đầu hàng: 2 | cuối hàng: 1
giữa: [2]


### 3.3. Chia nguyên `//` so với chia thực `/`

Khoảng cách là số nguyên. `(b - a) // 2` làm tròn xuống: với `b - a = 3` được `1` (ghế chính giữa không tồn tại, ghế nào cũng cách người gần nhất tối đa 1). Dùng `/` cho `1.5`: sai kiểu.

In [4]:
for gap in (2, 3, 4, 5):
    print(f"hai người cách nhau {gap}: ghế giữa tốt nhất cách {gap // 2} (không phải {gap / 2})")

hai người cách nhau 2: ghế giữa tốt nhất cách 1 (không phải 1.0)
hai người cách nhau 3: ghế giữa tốt nhất cách 1 (không phải 1.5)
hai người cách nhau 4: ghế giữa tốt nhất cách 2 (không phải 2.0)
hai người cách nhau 5: ghế giữa tốt nhất cách 2 (không phải 2.5)


### 3.4. `groupby` nhóm các số liên tiếp bằng nhau

`itertools.groupby(seats)` gom các phần tử liên tiếp giống nhau; ta lấy độ dài của các nhóm số 0 và vị trí của chúng (đầu / cuối / giữa).

In [5]:
seats = [0, 0, 1, 0, 0, 0, 1, 0]
pos = 0
for value, grp in groupby(seats):
    length = len(list(grp))
    print(f"nhóm giá trị {value}: bắt đầu {pos}, dài {length}")
    pos += length

nhóm giá trị 0: bắt đầu 0, dài 2
nhóm giá trị 1: bắt đầu 2, dài 1
nhóm giá trị 0: bắt đầu 3, dài 3
nhóm giá trị 1: bắt đầu 6, dài 1
nhóm giá trị 0: bắt đầu 7, dài 1


### 3.5. Chặt nhị phân trên đáp án và tổng tiền tố

Nếu có thể đạt khoảng cách `d` thì cũng đạt được `d - 1` (tính **đơn điệu**), nên có thể chặt nhị phân trên `d`. Để kiểm tra "ghế `i` cách mọi người ít nhất `d`" nhanh, ta cần đếm số người trong đoạn `[i - d + 1, i + d - 1]` bằng **tổng tiền tố**: `pre[r + 1] - pre[l]`.

In [6]:
seats = [1, 0, 0, 0, 1, 0, 1]
pre = [0] + list(accumulate(seats))
print("pre =", pre)
def people_in(l, r):
    l, r = max(l, 0), min(r, len(seats) - 1)
    return pre[r + 1] - pre[l]
print("số người trong [1, 3]:", people_in(1, 3), "| trong [1, 2]:", people_in(1, 2))

pre = [0, 1, 1, 1, 1, 2, 2, 3]
số người trong [1, 3]: 0 | trong [1, 2]: 0


## 4. Ý tưởng nền tảng / trực giác (tính tay)

`seats = [1,0,0,0,1,0,1]`. Các khoảng trống:

- Đầu hàng: không có (ghế 0 có người).
- Giữa người ở 0 và 4: khoảng cách `4 - 0 = 4`, ghế giữa cách `4 // 2 = 2`.
- Giữa người ở 4 và 6: `6 - 4 = 2`, ghế giữa cách `1`.
- Cuối hàng: không có.

Đáp án `max(0, 2, 1, 0) = 2`.

`seats = [1,0,0,0]`: người duy nhất ở 0. Cuối hàng: `n - 1 - 0 = 3` (ngồi ghế 3). Không có khoảng trống đầu hoặc giữa. Đáp án 3.

Hai đầu khác khoảng trống giữa ở chỗ **chỉ có một người bên cạnh**, nên ta ngồi sát mép xa người đó nhất.

In [7]:
def explain(seats):
    n = len(seats)
    people = [j for j, c in enumerate(seats) if c]
    print(f"seats={seats}, người ở {people}")
    head, tail = people[0], n - 1 - people[-1]
    mids = [((b - a) // 2, a, b) for a, b in zip(people, people[1:])]
    print(f"  đầu hàng: {head}; cuối hàng: {tail}; giữa: {[(m, f'giữa {a} và {b}') for m, a, b in mids]}")
    return max([head, tail] + [m for m, _, _ in mids])

for seats, expected in examples:
    assert explain(seats) == expected

seats=[1, 0, 0, 0, 1, 0, 1], người ở [0, 4, 6]
  đầu hàng: 0; cuối hàng: 0; giữa: [(2, 'giữa 0 và 4'), (1, 'giữa 4 và 6')]
seats=[1, 0, 0, 0], người ở [0]
  đầu hàng: 0; cuối hàng: 3; giữa: []
seats=[0, 1], người ở [1]
  đầu hàng: 1; cuối hàng: 0; giữa: []


## 5. Cách 1 — Brute Force: thử từng ghế trống

**Ý tưởng.** Với mỗi ghế trống `i`, tính khoảng cách tới người gần nhất bằng cách duyệt mọi người, rồi lấy `max` trên các ghế trống. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [8]:
def dist_brute(seats: List[int]) -> int:
    people = [j for j, c in enumerate(seats) if c]
    best = 0
    for i, c in enumerate(seats):
        if c == 0:
            best = max(best, min(abs(i - j) for j in people))
    return best

for seats, expected in examples:
    got = dist_brute(seats)
    print(seats, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 0, 0, 1, 0, 1] -> 2 OK
[1, 0, 0, 0] -> 3 OK
[0, 1] -> 1 OK


**Độ phức tạp và nút thắt.** Với `n` ghế và `p` người: `O(n * p)`, tệ nhất `O(n^2 / 4)` khi nửa số ghế có người. Với `n = 2 * 10^4` là `10^8`: quá chậm trong Python. Nút thắt: ta tính lại khoảng cách tới **toàn bộ** người cho mỗi ghế, dù chỉ cần hai người **kề** ghế đó (gần nhất bên trái và bên phải).

In [9]:
rng = random.Random(1)
for n in (500, 1000, 2000, 4000):
    seats = [1 if rng.random() < 0.5 else 0 for _ in range(n)]
    seats[0], seats[-1] = 1, 0
    t0 = time.perf_counter()
    dist_brute(seats)
    print(f"n={n:5}: vét cạn mất {1000 * (time.perf_counter() - t0):8.1f} ms")

n=  500: vét cạn mất      3.5 ms
n= 1000: vét cạn mất     14.2 ms
n= 2000: vét cạn mất     61.6 ms


n= 4000: vét cạn mất    288.3 ms


## 6. Ý tưởng thuật toán / bước đột phá

Với ghế trống `i`, người gần nhất luôn là **người gần nhất bên trái** hoặc **người gần nhất bên phải** (nếu có). Vậy khoảng cách = `min(dL[i], dR[i])` với `dL[i]` = khoảng cách tới người gần nhất bên trái (hoặc `∞` nếu không có), `dR[i]` tương tự bên phải. Hai mảng này tính trong **một lượt mỗi chiều**.

Hơn nữa, chỉ cần quan tâm tới **các khoảng trống** thay vì từng ghế, nhờ ba loại ở mục 3.2: đáp án `= max(người đầu tiên, n - 1 - người cuối, max khoảng giữa // 2)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Hai lượt: `dL`, `dR`, rồi `max(min(dL, dR))` trên ghế trống |
| 3 | Một lượt: theo dõi người đầu, người cuối, khoảng giữa lớn nhất |
| 4 | Nhóm các số 0 bằng `groupby`: đầu/cuối lấy nguyên độ dài, giữa lấy `(độ dài + 1) // 2` |
| 5 | Chặt nhị phân trên đáp án `d` với kiểm tra tổng tiền tố |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai lượt tính khoảng cách trái/phải

**Giải thích từng đoạn.**

- Lượt trái -> phải: `left[i]` = số ghế kể từ người gần nhất bên trái tới `i` (`INF` nếu chưa gặp người). Gặp người thì đặt `0` và nhớ vị trí; ghế trống thì `i - vị_trí_gần_nhất_trái`.
- Lượt phải -> trái: tương tự cho `right[i]`.
- Với ghế trống: `min(left[i], right[i])`; đáp án là max.

In [10]:
def dist_two_pass(seats: List[int]) -> int:
    n = len(seats)
    INF = n + 1
    left, right = [INF] * n, [INF] * n
    last = None
    for i in range(n):
        if seats[i]:
            last = i
        elif last is not None:
            left[i] = i - last
    last = None
    for i in range(n - 1, -1, -1):
        if seats[i]:
            last = i
        elif last is not None:
            right[i] = last - i
    return max(min(left[i], right[i]) for i in range(n) if seats[i] == 0)

for seats, expected in examples:
    assert dist_two_pass(seats) == expected
print("hai lượt OK")

hai lượt OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        n = len(seats)
        INF = n + 1
        left, right = [INF] * n, [INF] * n
        last = None
        for i in range(n):
            if seats[i]:
                last = i
            elif last is not None:
                left[i] = i - last
        last = None
        for i in range(n - 1, -1, -1):
            if seats[i]:
                last = i
            elif last is not None:
                right[i] = last - i
        return max(min(left[i], right[i]) for i in range(n) if seats[i] == 0)

sol_two = Solution().maxDistToClosest
print([sol_two(s) for s, _ in examples])

[2, 3, 1]


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(n)` cho hai mảng.

---

### 7.2. Cách 3 — Một lượt, không mảng phụ (lời giải chuẩn)

**Giải thích từng đoạn.**

- `first`, `last`: vị trí người đầu tiên và người gần nhất đã gặp (ban đầu `None`); `d`: khoảng cách lớn nhất giữa **hai người liên tiếp**.
- Gặp người ở `i`: nếu đã có `last` thì `d = max(d, i - last)`; nếu chưa có `first` thì `first = i`; cập nhật `last = i`.
- Kết quả `max(first, n - 1 - last, d // 2)`: ghế sát đầu trái (cách `first`), sát cuối phải (cách `n - 1 - last`), hoặc giữa hai người (`d // 2`).

In [12]:
def dist_single_pass(seats: List[int]) -> int:
    first = last = None
    d = 0
    for i, c in enumerate(seats):
        if c:
            if last is not None:
                d = max(d, i - last)
            if first is None:
                first = i
            last = i
    return max(first, len(seats) - 1 - last, d // 2)

for seats, expected in examples:
    assert dist_single_pass(seats) == expected
print("một lượt OK")

một lượt OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        first = last = None
        d = 0
        for i, c in enumerate(seats):
            if c:
                if last is not None:
                    d = max(d, i - last)
                if first is None:
                    first = i
                last = i
        return max(first, len(seats) - last - 1, d // 2)

sol_single = Solution().maxDistToClosest
print([sol_single(s) for s, _ in examples])

[2, 3, 1]


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(1)`. **Đây là cách nên nộp.**

---

### 7.3. Cách 4 — Nhóm các số 0 bằng `groupby`

**Ý tưởng khác.** Mỗi nhóm số 0 liên tiếp là một khoảng trống độ dài `L`. Tùy vị trí:

- sát đầu mảng hoặc sát cuối mảng: đóng góp `L` (ngồi ở mép);
- ở giữa hai người: đóng góp `(L + 1) // 2` (khoảng trống `L` ghế nghĩa là hai người cách nhau `L + 1`, ghế giữa cách `(L + 1) // 2`).

**Giải thích từng đoạn.** Duyệt `groupby`; với nhóm số 0, biết vị trí bắt đầu và kết thúc để biết nhóm chạm đầu/cuối hay không.

In [14]:
def dist_groupby(seats: List[int]) -> int:
    n = len(seats)
    best = 0
    pos = 0
    for value, grp in groupby(seats):
        L = len(list(grp))
        if value == 0:
            if pos == 0 or pos + L == n:
                best = max(best, L)                  # khoảng trống chạm mép
            else:
                best = max(best, (L + 1) // 2)       # khoảng trống giữa hai người
        pos += L
    return best

for seats, expected in examples:
    assert dist_groupby(seats) == expected
print("groupby OK")

groupby OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        from itertools import groupby
        n = len(seats)
        ans = pos = 0
        for value, grp in groupby(seats):
            L = len(list(grp))
            if value == 0:
                if pos == 0 or pos + L == n:
                    ans = max(ans, L)
                else:
                    ans = max(ans, (L + 1) // 2)
            pos += L
        return ans

sol_groupby = Solution().maxDistToClosest
print([sol_groupby(s) for s, _ in examples])

[2, 3, 1]


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(1)` (mỗi nhóm tạm thời một list ngắn, tổng `O(n)` thời gian).

---

### 7.4. Cách 5 — Chặt nhị phân trên đáp án

**Ý tưởng khác.** Hỏi "có ghế trống nào cách **mọi** người ít nhất `d` không?". Ghế `i` thỏa nếu không có người trong `[i - d + 1, i + d - 1]` (khoảng cách `< d` nghĩa là `|i - j| <= d - 1`). Tính đơn điệu: nếu `d` đạt được thì `d - 1` cũng đạt; nên chặt nhị phân tìm `d` lớn nhất, `1 <= d <= n - 1`.

**Giải thích từng đoạn.**

- `pre` là tổng tiền tố của `seats`; `count(l, r)` là số người trong đoạn (cắt biên).
- `feasible(d)`: có `i` với `seats[i] == 0` và `count(i - d + 1, i + d - 1) == 0`.
- `lo = 1` luôn đạt (có ít nhất một ghế trống), `hi = n - 1`.

In [16]:
def dist_binary_search(seats: List[int]) -> int:
    n = len(seats)
    pre = [0] + list(accumulate(seats))

    def count(l: int, r: int) -> int:
        l, r = max(l, 0), min(r, n - 1)
        return pre[r + 1] - pre[l]

    def feasible(d: int) -> bool:
        return any(seats[i] == 0 and count(i - d + 1, i + d - 1) == 0 for i in range(n))

    lo, hi = 1, n - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for seats, expected in examples:
    assert dist_binary_search(seats) == expected
print("chặt nhị phân OK")

chặt nhị phân OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        from itertools import accumulate
        n = len(seats)
        pre = [0] + list(accumulate(seats))

        def count(l: int, r: int) -> int:
            l, r = max(l, 0), min(r, n - 1)
            return pre[r + 1] - pre[l]

        def feasible(d: int) -> bool:
            return any(seats[i] == 0 and count(i - d + 1, i + d - 1) == 0 for i in range(n))

        lo, hi = 1, n - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if feasible(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

sol_binary = Solution().maxDistToClosest
print([sol_binary(s) for s, _ in examples])

[2, 3, 1]


**Độ phức tạp (Cách 5):** thời gian `O(n log n)` (khoảng `log n` lần `feasible`, mỗi lần `O(n)`), bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 3 trên `[1, 0, 0, 0, 1, 0, 1]`

| `i` | `seats[i]` | `last` trước | `first` | `d` sau | `last` sau |
|---|---|---|---|---|---|
| 0 | 1 | None | 0 | 0 | 0 |
| 1..3 | 0 | | | | |
| 4 | 1 | 0 | 0 | `max(0, 4 - 0) = 4` | 4 |
| 5 | 0 | | | | |
| 6 | 1 | 4 | 0 | `max(4, 6 - 4) = 4` | 6 |

Kết quả `max(first = 0, n - 1 - last = 0, d // 2 = 2) = 2`.

In [18]:
def trace_single(seats):
    first = last = None
    d = 0
    print("seats =", seats)
    for i, c in enumerate(seats):
        if c:
            old_last = last
            if last is not None:
                d = max(d, i - last)
            if first is None:
                first = i
            last = i
            print(f"  i={i}: có người; last trước = {old_last}, first = {first}, d = {d}, last = {last}")
    n = len(seats)
    head, tail, mid = first, n - 1 - last, d // 2
    print(f"  đầu hàng = {head}, cuối hàng = {tail}, giữa = d // 2 = {mid} -> max = {max(head, tail, mid)}")
    return max(head, tail, mid)

assert trace_single([1, 0, 0, 0, 1, 0, 1]) == 2
print()
assert trace_single([1, 0, 0, 0]) == 3
print()
assert trace_single([0, 0, 1, 0, 0, 0, 0, 1, 0]) == 2

seats = [1, 0, 0, 0, 1, 0, 1]
  i=0: có người; last trước = None, first = 0, d = 0, last = 0
  i=4: có người; last trước = 0, first = 0, d = 4, last = 4
  i=6: có người; last trước = 4, first = 0, d = 4, last = 6
  đầu hàng = 0, cuối hàng = 0, giữa = d // 2 = 2 -> max = 2

seats = [1, 0, 0, 0]
  i=0: có người; last trước = None, first = 0, d = 0, last = 0
  đầu hàng = 0, cuối hàng = 3, giữa = d // 2 = 0 -> max = 3

seats = [0, 0, 1, 0, 0, 0, 0, 1, 0]
  i=2: có người; last trước = None, first = 2, d = 0, last = 2
  i=7: có người; last trước = 2, first = 2, d = 5, last = 7
  đầu hàng = 2, cuối hàng = 1, giữa = d // 2 = 2 -> max = 2


### 8.2. Cách 2: mảng `left`, `right`, và `min` trên `[1, 0, 0, 0, 1, 0, 1]`

| `i` | `seats[i]` | `left[i]` | `right[i]` | `min` |
|---|---|---|---|---|
| 1 | 0 | 1 | 3 | 1 |
| 2 | 0 | 2 | 2 | **2** |
| 3 | 0 | 3 | 1 | 1 |
| 5 | 0 | 1 | 1 | 1 |

In [19]:
def trace_two_pass(seats):
    n = len(seats)
    INF = n + 1
    left, right = [INF] * n, [INF] * n
    last = None
    for i in range(n):
        if seats[i]:
            last = i
        elif last is not None:
            left[i] = i - last
    last = None
    for i in range(n - 1, -1, -1):
        if seats[i]:
            last = i
        elif last is not None:
            right[i] = last - i
    print("seats:", seats)
    print("left :", [x if x != INF else "inf" for x in left])
    print("right:", [x if x != INF else "inf" for x in right])
    vals = {i: min(left[i], right[i]) for i in range(n) if seats[i] == 0}
    print("min  :", vals)
    return max(vals.values())

assert trace_two_pass([1, 0, 0, 0, 1, 0, 1]) == 2
print()
assert trace_two_pass([0, 0, 1, 0]) == 2

seats: [1, 0, 0, 0, 1, 0, 1]
left : ['inf', 1, 2, 3, 'inf', 1, 'inf']
right: ['inf', 3, 2, 1, 'inf', 1, 'inf']
min  : {1: 1, 2: 2, 3: 1, 5: 1}

seats: [0, 0, 1, 0]
left : ['inf', 'inf', 'inf', 1]
right: [2, 1, 'inf', 'inf']
min  : {0: 2, 1: 1, 3: 1}


### 8.3. Cách 4: các nhóm số 0 của `[0, 0, 1, 0, 0, 0, 1, 0]`

Nhóm số 0 đầu (dài 2, chạm mép trái): đóng góp 2. Nhóm giữa (dài 3, giữa hai người): `(3 + 1) // 2 = 2`. Nhóm cuối (dài 1, chạm mép phải): 1. Đáp án `max(2, 2, 1) = 2`.

In [20]:
def trace_groupby(seats):
    n = len(seats)
    pos = 0
    best = 0
    for value, grp in groupby(seats):
        L = len(list(grp))
        if value == 0:
            kind = "mép" if (pos == 0 or pos + L == n) else "giữa"
            contrib = L if kind == "mép" else (L + 1) // 2
            best = max(best, contrib)
            print(f"  nhóm 0 tại [{pos}, {pos + L - 1}], dài {L}, loại {kind} -> đóng góp {contrib}")
        pos += L
    print("kết quả =", best)
    return best

assert trace_groupby([0, 0, 1, 0, 0, 0, 1, 0]) == 2

  nhóm 0 tại [0, 1], dài 2, loại mép -> đóng góp 2
  nhóm 0 tại [3, 5], dài 3, loại giữa -> đóng góp 2
  nhóm 0 tại [7, 7], dài 1, loại mép -> đóng góp 1
kết quả = 2


### 8.4. Cách 5: chặt nhị phân cho `[1, 0, 0, 0, 1, 0, 1]`

Khoảng `[1, 6]`. `mid = 4`: có ghế nào cách mọi người >= 4? không. `hi = 3`. `mid = 2`: ghế 2 thỏa. `lo = 2`. `mid = 3`: không. `hi = 2`. Kết quả `2`.

In [21]:
def trace_binary(seats):
    n = len(seats)
    pre = [0] + list(accumulate(seats))
    def count(l, r):
        l, r = max(l, 0), min(r, n - 1)
        return pre[r + 1] - pre[l]
    def feasible(d):
        for i in range(n):
            if seats[i] == 0 and count(i - d + 1, i + d - 1) == 0:
                return True, i
        return False, None
    lo, hi = 1, n - 1
    print(f"khoảng ban đầu [{lo}, {hi}]")
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok, where = feasible(mid)
        print(f"  thử d = {mid}: {'có, ghế ' + str(where) if ok else 'không'}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
        print(f"  -> [{lo}, {hi}]")
    return lo

assert trace_binary([1, 0, 0, 0, 1, 0, 1]) == 2

khoảng ban đầu [1, 6]
  thử d = 4: không
  -> [1, 3]
  thử d = 2: có, ghế 2
  -> [2, 3]
  thử d = 3: không
  -> [2, 2]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (một lượt): `O(n)` thời gian, `O(1)` bộ nhớ, ba biến.

```python
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        first = last = None
        d = 0
        for i, c in enumerate(seats):
            if c:
                if last is not None:
                    d = max(d, i - last)
                if first is None:
                    first = i
                last = i
        return max(first, len(seats) - last - 1, d // 2)
```

In [22]:
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        first = last = None
        d = 0
        for i, c in enumerate(seats):
            if c:
                if last is not None:
                    d = max(d, i - last)
                if first is None:
                    first = i
                last = i
        return max(first, len(seats) - last - 1, d // 2)

sol = Solution()
for seats, expected in examples:
    got = sol.maxDistToClosest(seats)
    print(seats, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 0, 0, 0, 1, 0, 1] -> 2 OK
[1, 0, 0, 0] -> 3 OK
[0, 1] -> 1 OK


## 10. Vì sao đúng

**(a) Chỉ cần hai người kề.** Với ghế trống `i`, người gần nhất là người gần nhất bên trái hoặc người gần nhất bên phải (người xa hơn ở cùng phía chỉ cách xa hơn). Do đó ghế `i` thuộc **một** khoảng trống và khoảng cách của nó chỉ phụ thuộc hai biên của khoảng trống đó.

**(b) Giá trị tốt nhất trong một khoảng trống.**

- Khoảng trống **giữa hai người** ở `a < b` (các ghế `a + 1 .. b - 1` trống): ghế `i` có khoảng cách `min(i - a, b - i)`. Hàm này lớn nhất ở giữa: `max_i min(i - a, b - i) = (b - a) // 2` (đạt tại `i = a + (b - a) // 2`).
- Khoảng trống **đầu mảng** (các ghế `0 .. p - 1` trống, `p` là người đầu tiên): chỉ có người bên phải, khoảng cách `p - i` lớn nhất khi `i = 0`: giá trị `p`.
- Khoảng trống **cuối mảng**: đối xứng, giá trị `n - 1 - q` với `q` là người cuối.

Đáp án là max các giá trị trên mọi khoảng trống; trong code `d` là khoảng cách lớn nhất giữa hai người **liên tiếp**, và `d // 2` là max các `(b - a) // 2` (hàm tăng theo `b - a`). Khoảng trống rỗng (hai người kề nhau, `b - a = 1`) cho `1 // 2 = 0`, không ảnh hưởng.

**(c) Cách 2.** `left[i]`, `right[i]` là khoảng cách tới người gần nhất mỗi bên; `min` của chúng là khoảng cách tới người gần nhất, đúng định nghĩa.

**(d) Cách 4.** Khoảng trống giữa dài `L` ghế nghĩa là `b - a = L + 1`, nên đóng góp `(L + 1) // 2`; khoảng trống chạm mép dài `L` cho `L`.

**(e) Cách 5.** `feasible(d)` đúng vì "mọi người cách `i` ít nhất `d`" ⇔ "không có người nào trong `[i - d + 1, i + d - 1]`". Tính đơn điệu: nếu `d` đạt tại ghế `i` thì `d - 1` cũng đạt tại `i`. Đáp án `>= 1` luôn (có ghế trống).

**Kiểm chứng bằng máy:** so mọi cách với oracle vét cạn trên **mọi** mảng nhị phân độ dài 2..16 thỏa ràng buộc (có ít nhất một người và một ghế trống).

In [23]:
solutions = {
    "brute": dist_brute,
    "two_pass": dist_two_pass,
    "single": dist_single_pass,
    "groupby": dist_groupby,
    "binary": dist_binary_search,
    "sol_two": sol_two,
    "sol_single": sol_single,
    "sol_groupby": sol_groupby,
    "sol_binary": sol_binary,
    "final": Solution().maxDistToClosest,
}

checked = 0
for n in range(2, 17):
    for mask in range(1, (1 << n) - 1):                 # bỏ mask toàn 0 và toàn 1
        seats = [mask >> i & 1 for i in range(n)]
        ref = dist_brute(seats)
        for name, fn in solutions.items():
            assert fn(seats) == ref, (name, seats, fn(seats), ref)
        checked += 1
print(f"mọi mảng nhị phân độ dài 2..16 hợp lệ: {checked} mảng x {len(solutions)} cài đặt khớp oracle")

mọi mảng nhị phân độ dài 2..16 hợp lệ: 131038 mảng x 10 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n = len(seats)` (`<= 2 * 10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn từng ghế | `O(n * p)`, tệ nhất `O(n^2)` | `O(p)` |
| 2. Hai lượt | `O(n)` | `O(n)` |
| 3. Một lượt | `O(n)` | `O(1)` |
| 4. `groupby` | `O(n)` | `O(1)` |
| 5. Chặt nhị phân | `O(n log n)` | `O(n)` |

Đo với `n = 20000` (bỏ vét cạn cho các mảng nhiều người):

In [24]:
rng = random.Random(849)
big = []
big.append([1] + [0] * 19998 + [1])                                    # một khoảng trống khổng lồ giữa
big.append([0] * 19999 + [1])                                          # toàn bộ trống ở đầu
big.append([1] + [0] * 19999)                                          # toàn bộ trống ở cuối
big.append([1, 0] * 10000)                                             # xen kẽ
s = [1 if rng.random() < 0.01 else 0 for _ in range(20000)]
s[10000] = 1
s[5000] = 0
big.append(s)                                                          # thưa

ref = None
for name in ["two_pass", "single", "groupby", "binary"]:
    t0 = time.perf_counter()
    res = [solutions[name](x) for x in big]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:9} {len(big)} mảng n=20000: {dt:8.1f} ms")
print("kết quả:", ref)
t0 = time.perf_counter()
small = [1] + [0] * 2998 + [1]
assert dist_brute(small) == Solution().maxDistToClosest(small) == 1499
print(f"brute n=3000 (khoảng trống lớn): {1000 * (time.perf_counter() - t0):.1f} ms")

two_pass  5 mảng n=20000:     26.0 ms
single    5 mảng n=20000:      5.3 ms
groupby   5 mảng n=20000:      8.5 ms


binary    5 mảng n=20000:    337.0 ms
kết quả: [9999, 19999, 19999, 1, 291]
brute n=3000 (khoảng trống lớn): 2.6 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle vét cạn và test ngẫu nhiên có seed (đúng ràng buộc: `2 <= n <= 2 * 10^4`, có ít nhất một người và một ghế trống).

In [25]:
# 12.1 ví dụ đề
for seats, expected in examples:
    for name, fn in solutions.items():
        assert fn(seats) == expected, (name, seats)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0, 1], 1),                                    # người ở cuối: ngồi ghế 0
    ([1, 0], 1),                                    # người ở đầu: ngồi ghế 1
    ([1, 0, 1], 1),                                 # ghế giữa hai người
    ([1, 0, 0, 1], 1),                              # hai ghế trống giữa: mỗi ghế cách 1
    ([1, 0, 0, 0, 1], 2),                           # ghế giữa cách 2
    ([1, 0, 0, 0, 0, 1], 2),                        # 4 ghế trống giữa: (5 // 2) = 2
    ([0, 0, 1], 2),                                 # đầu hàng: ngồi ghế 0 cách 2
    ([0, 0, 0, 1, 0], 3),                           # đầu hàng 3 ghế trống > cuối 1
    ([1, 0, 0, 0, 0], 4),                           # cuối hàng
    ([0, 1, 0, 0, 0, 1, 0], 2),                     # giữa cách 2 > mép 1
    ([0, 0, 1, 0, 0, 0, 0, 1, 0, 0], 2),           # đầu 2, giữa 5//2 = 2, cuối 2
    ([0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0], 3),         # đầu 3, giữa 6//2 = 3, cuối 1
    ([1] + [0] * 19998 + [1], 9999),                # lớn nhất: (19999) // 2
    ([0] * 19999 + [1], 19999),                     # toàn bộ ở đầu
    ([1] + [0] * 19999, 19999),                     # toàn bộ ở cuối
    ([1, 0] * 10000, 1),                            # xen kẽ
]
for seats, expected in edge:
    if len(seats) <= 3000:
        assert dist_brute(seats) == expected, ("oracle khác kỳ vọng", seats, dist_brute(seats))
    for name, fn in solutions.items():
        if name == "brute" and len(seats) > 3000:
            continue
        assert fn(seats) == expected, (name, seats[:10], fn(seats), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 16 ca


In [26]:
# 12.3 phản ví dụ: các cách sai
def wrong_only_middle(seats):          # bỏ qua hai đầu
    people = [j for j, c in enumerate(seats) if c]
    return max([(b - a) // 2 for a, b in zip(people, people[1:])] + [0])

def wrong_ends_halved(seats):          # dùng // 2 cho cả hai đầu
    first = last = None
    d = 0
    for i, c in enumerate(seats):
        if c:
            if last is not None:
                d = max(d, i - last)
            if first is None:
                first = i
            last = i
    return max(first // 2, (len(seats) - 1 - last) // 2, d // 2)

def wrong_gap_off_by_one(seats):       # d - 1 // 2 cho khoảng giữa
    first = last = None
    d = 0
    for i, c in enumerate(seats):
        if c:
            if last is not None:
                d = max(d, i - last)
            if first is None:
                first = i
            last = i
    return max(first, len(seats) - 1 - last, (d - 1) // 2)

def wrong_zero_count_half(seats):      # lấy số ghế trống / 2 thay vì (độ cách) / 2
    best, run = 0, 0
    first_person_seen = False
    for c in seats:
        if c:
            best = max(best, run // 2 if first_person_seen else run)
            first_person_seen = True
            run = 0
        else:
            run += 1
    return max(best, run)

def wrong_ends_minus_one(seats):       # đầu/cuối trừ 1
    first = last = None
    d = 0
    for i, c in enumerate(seats):
        if c:
            if last is not None:
                d = max(d, i - last)
            if first is None:
                first = i
            last = i
    return max(first - 1, len(seats) - 2 - last, d // 2)

wrongs = [
    ("bỏ qua hai đầu", wrong_only_middle, [1, 0, 0, 0]),
    ("hai đầu chia đôi", wrong_ends_halved, [0, 0, 0, 1]),
    ("khoảng giữa lệch 1", wrong_gap_off_by_one, [1, 0, 0, 0, 1]),
    ("đếm số ghế trống chia đôi", wrong_zero_count_half, [1, 0, 1]),
    ("đầu/cuối trừ 1", wrong_ends_minus_one, [1, 0, 0, 0]),
]
for name, fn, seats in wrongs:
    truth = dist_brute(seats)
    print(f"{name:28} seats={seats}: sai = {fn(seats)}, đúng = {truth}")
    assert fn(seats) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

bỏ qua hai đầu               seats=[1, 0, 0, 0]: sai = 0, đúng = 3
hai đầu chia đôi             seats=[0, 0, 0, 1]: sai = 1, đúng = 3
khoảng giữa lệch 1           seats=[1, 0, 0, 0, 1]: sai = 1, đúng = 2
đếm số ghế trống chia đôi    seats=[1, 0, 1]: sai = 0, đúng = 1
đầu/cuối trừ 1               seats=[1, 0, 0, 0]: sai = 2, đúng = 3
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [27]:
# 12.4 ngẫu nhiên nhỏ so với oracle vét cạn
rng = random.Random(2849)
for _ in range(1500):
    n = rng.randint(2, 40)
    density = rng.choice([0.05, 0.2, 0.5, 0.9])
    seats = [1 if rng.random() < density else 0 for _ in range(n)]
    if sum(seats) == 0:
        seats[rng.randrange(n)] = 1
    if sum(seats) == n:
        seats[rng.randrange(n)] = 0
    ref = dist_brute(seats)
    for name, fn in solutions.items():
        assert fn(seats) == ref, (name, seats)
print("12.4 ngẫu nhiên nhỏ so với oracle: OK")

# 12.5 ngẫu nhiên lớn (n tới 20000): các cách nhanh khớp nhau; đối chiếu brute khi n <= 3000
rng = random.Random(77)
for _ in range(60):
    n = rng.randint(1000, 20000)
    density = rng.choice([0.001, 0.01, 0.1, 0.5])
    seats = [1 if rng.random() < density else 0 for _ in range(n)]
    if sum(seats) == 0:
        seats[rng.randrange(n)] = 1
    if sum(seats) == n:
        seats[rng.randrange(n)] = 0
    res = {name: fn(seats) for name, fn in solutions.items() if name != "brute"}
    assert len(set(res.values())) == 1, (n, density, res)
    if n <= 3000:
        assert res["final"] == dist_brute(seats)
print("12.5 ngẫu nhiên lớn: OK")

# 12.6 tính chất: đáp án <= n - 1, và đảo ngược mảng không đổi đáp án
rng = random.Random(6)
for _ in range(300):
    n = rng.randint(2, 60)
    seats = [rng.randint(0, 1) for _ in range(n)]
    if sum(seats) in (0, n):
        continue
    ans = Solution().maxDistToClosest(seats)
    assert 1 <= ans <= n - 1
    assert ans == Solution().maxDistToClosest(seats[::-1])
print("12.6 tính chất (đối xứng, cận): OK")

12.4 ngẫu nhiên nhỏ so với oracle: OK


12.5 ngẫu nhiên lớn: OK
12.6 tính chất (đối xứng, cận): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua hai đầu hàng ghế

Chỉ xét khoảng giữa hai người, quên rằng ngồi sát mép có thể xa người hơn.

In [28]:
seats = [1, 0, 0, 0]
print("sai (chỉ khoảng giữa):", wrong_only_middle(seats), "| đúng:", Solution().maxDistToClosest(seats))
assert wrong_only_middle(seats) == 0 and Solution().maxDistToClosest(seats) == 3

sai (chỉ khoảng giữa): 0 | đúng: 3


Vì sao sai: không có hai người nào nên không có khoảng giữa, nhưng ghế cuối cách người 3. **Đúng:** thêm `first` và `n - 1 - last`.

### Lỗi 2 — Chia đôi cả hai đầu

Ở mép chỉ có **một** người bên cạnh nên **không** có điểm giữa để chia đôi.

In [29]:
seats = [0, 0, 0, 1]
print("sai (đầu // 2):", wrong_ends_halved(seats), "| đúng:", Solution().maxDistToClosest(seats))
assert wrong_ends_halved(seats) == 1 and Solution().maxDistToClosest(seats) == 3

sai (đầu // 2): 1 | đúng: 3


Vì sao sai: ngồi ghế 0 cách người ở ghế 3 tới 3 ghế. **Đúng:** hai đầu lấy nguyên độ dài, chỉ khoảng giữa mới `// 2`.

### Lỗi 3 — Lệch 1 ở khoảng giữa

Hai người cách nhau `d = b - a` (chứ không phải `d - 1`): ghế giữa cách `d // 2`.

In [30]:
seats = [1, 0, 0, 0, 1]
print("sai ((d - 1) // 2):", wrong_gap_off_by_one(seats), "| đúng:", Solution().maxDistToClosest(seats))
assert wrong_gap_off_by_one(seats) == 1 and Solution().maxDistToClosest(seats) == 2

sai ((d - 1) // 2): 1 | đúng: 2


Vì sao sai: `[1,0,0,0,1]` có `d = 4`; `(4 - 1) // 2 = 1` nhưng ghế giữa cách 2 (`4 // 2`). **Đúng:** `d // 2` với `d` là khoảng cách **chỉ số** hai người.

### Lỗi 4 — Dùng số ghế trống chia đôi

Khoảng trống `L` ghế ứng với hai người cách `L + 1`: đóng góp đúng là `(L + 1) // 2`, không phải `L // 2`.

In [31]:
seats = [1, 0, 1]
print("sai (L // 2):", wrong_zero_count_half(seats), "| đúng:", Solution().maxDistToClosest(seats))
assert wrong_zero_count_half(seats) == 0 and Solution().maxDistToClosest(seats) == 1

sai (L // 2): 0 | đúng: 1


Vì sao sai: `[1, 0, 1]` có một ghế trống (`L = 1`): ghế đó cách mỗi người 1, nhưng `1 // 2 = 0`. **Đúng:** `(L + 1) // 2` hoặc dùng khoảng cách chỉ số.

### Lỗi 5 — Trừ 1 ở hai đầu

Khoảng cách tới người đầu tiên (chỉ số `p`) khi ngồi ghế 0 là **`p`**, không phải `p - 1`.

In [32]:
seats = [1, 0, 0, 0]
print("sai (n - 2 - last):", wrong_ends_minus_one(seats), "| đúng:", Solution().maxDistToClosest(seats))
assert wrong_ends_minus_one(seats) == 2 and Solution().maxDistToClosest(seats) == 3

sai (n - 2 - last): 2 | đúng: 3


Vì sao sai: người ở ghế 0, ngồi ghế 3: cách `3 - 0 = 3`. Công thức `n - 1 - last` đã tính đúng; trừ thêm 1 là lệch. **Đúng:** `n - 1 - last`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "khoảng cách tới đối tượng gần nhất", "ghế/ô trống" -> nghĩ tới **khoảng trống giữa hai đối tượng** và **hai mép**.
- Quy tắc nhớ nhanh: **giữa `// 2`, mép nguyên vẹn**.
- Với "min khoảng cách tới thứ gần nhất" trên mảng: hai lượt (trái, phải) hoặc quét khoảng trống một lượt.
- **Chặt nhị phân trên đáp án** khi có tính đơn điệu và kiểm tra khả thi rẻ.
- Bài thiết kế mở rộng: Exam Room (855) — nhiều người vào/ra, dùng heap hoặc danh sách có thứ tự.
- Cẩn thận chia nguyên `//` (không `/`) và lệch một ở khoảng cách chỉ số.
- Test nên có: người ở đầu/cuối, hai người sát nhau, khoảng trống lớn ở đầu/cuối/giữa, hàng xen kẽ.

## 15. Biến thể

1. **Trả về ghế** (chỉ số) cho Alex thay vì khoảng cách.
2. **Đặt `k` người mới** sao cho khoảng cách nhỏ nhất tới người gần nhất (tính cả người mới) là lớn nhất (bài tập 5): chặt nhị phân + tham lam.
3. **Exam Room (855)**: nhiều lần ngồi và rời đi.
4. Bài liên quan: Maximize Distance to Closest Person, Minimum Distance Between BST Nodes, Heaters (475).

Dưới đây là code cho biến thể 1 (trả về một ghế tốt nhất) và kiểm bằng hàm kiểm tra tính hợp lệ (nhiều đáp án đúng).

In [33]:
def best_seat(seats: List[int]) -> int:
    n = len(seats)
    people = [j for j, c in enumerate(seats) if c]
    best_d, best_i = -1, -1
    if people[0] > 0 and people[0] > best_d:
        best_d, best_i = people[0], 0
    for a, b in zip(people, people[1:]):
        if (b - a) // 2 > best_d:
            best_d, best_i = (b - a) // 2, a + (b - a) // 2
    if n - 1 - people[-1] > best_d:
        best_d, best_i = n - 1 - people[-1], n - 1
    return best_i

def seat_distance(seats, i):
    return min(abs(i - j) for j, c in enumerate(seats) if c)

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(2, 30)
    seats = [1 if rng.random() < rng.choice([0.1, 0.3, 0.6]) else 0 for _ in range(n)]
    if sum(seats) in (0, n):
        continue
    i = best_seat(seats)
    assert seats[i] == 0 and seat_distance(seats, i) == Solution().maxDistToClosest(seats), (seats, i)
print("ghế tìm được hợp lệ và đạt khoảng cách tối ưu: OK")

ghế tìm được hợp lệ và đạt khoảng cách tối ưu: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mỗi ghế trống duyệt mọi người | `O(n p)` | `O(p)` | Oracle, `n` nhỏ |
| 2. Hai lượt | `left`, `right`, `min` | `O(n)` | `O(n)` | Dễ suy luận theo từng ghế |
| 3. Một lượt | Người đầu/cuối + khoảng giữa lớn nhất | `O(n)` | `O(1)` | **Cách nộp chính** |
| 4. `groupby` | Nhóm số 0: mép nguyên, giữa `(L+1)//2` | `O(n)` | `O(1)` | Trực quan theo "khoảng trống" |
| 5. Chặt nhị phân | Đơn điệu trên đáp án + tổng tiền tố | `O(n log n)` | `O(n)` | Khi mở rộng (nhiều người mới) |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(Solution().maxDistToClosest([0, 0, 1]))
print(Solution().maxDistToClosest([1, 0, 0, 0, 1]))
print([ (b - a) // 2 for a, b in zip([1, 5, 6], [5, 6, 12]) ])
```

In [34]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `seats = [0, 1, 0, 0, 0, 0, 1, 0, 0]`: ghi `first`, `n - 1 - last`, khoảng giữa lớn nhất (`d`) và đáp án.

In [35]:
# Bài 2: điền kết quả tính tay
hand_first = None
hand_tail = None
hand_d = None
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách một lượt.

In [36]:
def exercise_3_single(seats: List[int]) -> int:
    first = last = None
    d = 0
    for i, c in enumerate(seats):
        if c:
            # TODO 1: nếu đã có last thì cập nhật d = max(d, i - last)
            # TODO 2: nếu chưa có first thì first = i
            # TODO 3: last = i
            pass
    # TODO 4: trả về max của (first, n - 1 - last, d // 2)
    return None


# assert exercise_3_single([1, 0, 0, 0, 1, 0, 1]) == 2
# assert exercise_3_single([1, 0, 0, 0]) == 3
# assert exercise_3_single([0, 1]) == 1

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_two_pass(seats)` với hai mảng `left`, `right`.

In [37]:
def exercise_4_two_pass(seats: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_two_pass([1, 0, 0, 0, 1, 0, 1]) == 2
# assert exercise_4_two_pass([1, 0, 0, 0]) == 3
# assert exercise_4_two_pass([0, 1]) == 1

### Bài 5 — Mini challenge: thêm `k` người (khó)

Có thể thêm **`k` người mới** vào `k` ghế trống khác nhau. Gọi **chất lượng** là khoảng cách **nhỏ nhất** từ một người **mới** tới người gần nhất (tính cả người cũ và người mới khác). Trả về chất lượng **lớn nhất** có thể đạt. Biết có ít nhất `k` ghế trống.

```python
exercise_5_k_people([1, 0, 0, 0, 0, 0, 1], 1)  -> 3
exercise_5_k_people([1, 0, 0, 0, 0, 0, 1], 2)  -> 2     # ghế 2 và 4: cách nhau 2, cách người cũ 2
```
Gợi ý: chặt nhị phân trên `D`; tham lam từ trái sang phải: đặt người mới ở ghế trống đầu tiên cách mọi người đã có (cũ và mới) ít nhất `D`.

In [38]:
def exercise_5_k_people(seats: List[int], k: int) -> int:
    # TODO
    pass


# assert exercise_5_k_people([1, 0, 0, 0, 0, 0, 1], 1) == 3
# assert exercise_5_k_people([1, 0, 0, 0, 0, 0, 1], 2) == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `[0, 0, 1]`: người cuối cùng ở ghế 2.
- Mức 2: ngồi ghế 0 cách 2; `[1, 0, 0, 0, 1]`: hai người cách 4.
- Mức 3: `2`, `2`, `[2, 0, 3]`.

**Bài 2**
- Mức 1: người đầu ở chỉ số 1, người cuối ở chỉ số 6, `n = 9`.
- Mức 2: `first = 1`, `n - 1 - last = 2`, `d = 6 - 1 = 5`.
- Mức 3: `max(1, 2, 5 // 2 = 2) = 2`.

**Bài 3**
- Mức 1: `if last is not None: d = max(d, i - last)`.
- Mức 2: `if first is None: first = i`; `last = i`.
- Mức 3: `return max(first, len(seats) - 1 - last, d // 2)`.

**Bài 4**
- Mức 1: `left[i]` = `i - vị trí người gần nhất bên trái`, `INF` nếu chưa có.
- Mức 2: lượt phải -> trái cho `right[i]`.
- Mức 3: `max(min(left[i], right[i]) for i in range(n) if seats[i] == 0)`.

**Bài 5**
- Mức 1: kiểm tra `feasible(D)`: có đặt được `k` người với khoảng cách >= `D` không.
- Mức 2: tham lam: duyệt ghế từ trái sang phải, đặt khi cách mọi người trước đó (cũ + mới) ít nhất `D` về cả hai phía (người cũ bên phải cũng phải cách ít nhất `D`).
- Mức 3: chặt nhị phân `D` từ `1` tới `n`; kiểm tra bằng oracle `combinations`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [39]:
print(Solution().maxDistToClosest([0, 0, 1]))
print(Solution().maxDistToClosest([1, 0, 0, 0, 1]))
print([(b - a) // 2 for a, b in zip([1, 5, 6], [5, 6, 12])])

assert Solution().maxDistToClosest([0, 0, 1]) == 2
assert Solution().maxDistToClosest([1, 0, 0, 0, 1]) == 2
assert [(b - a) // 2 for a, b in zip([1, 5, 6], [5, 6, 12])] == [2, 0, 3]

2
2
[2, 0, 3]


### Đáp án Bài 2

`first = 1`, `n - 1 - last = 9 - 1 - 6 = 2`, `d = 6 - 1 = 5`, `d // 2 = 2`; đáp án `max(1, 2, 2) = 2`.

In [40]:
seats = [0, 1, 0, 0, 0, 0, 1, 0, 0]
hand_first_solution, hand_tail_solution, hand_d_solution, hand_answer_solution = 1, 2, 5, 2
people = [j for j, c in enumerate(seats) if c]
assert people[0] == hand_first_solution
assert len(seats) - 1 - people[-1] == hand_tail_solution
assert max(b - a for a, b in zip(people, people[1:])) == hand_d_solution
assert Solution().maxDistToClosest(seats) == hand_answer_solution == dist_brute(seats)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [41]:
def exercise_3_single_solution(seats: List[int]) -> int:
    first = last = None
    d = 0
    for i, c in enumerate(seats):
        if c:
            if last is not None:
                d = max(d, i - last)
            if first is None:
                first = i
            last = i
    return max(first, len(seats) - 1 - last, d // 2)

assert exercise_3_single_solution([1, 0, 0, 0, 1, 0, 1]) == 2
assert exercise_3_single_solution([1, 0, 0, 0]) == 3
assert exercise_3_single_solution([0, 1]) == 1
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(2, 30)
    seats = [rng.randint(0, 1) for _ in range(n)]
    if sum(seats) in (0, n):
        continue
    assert exercise_3_single_solution(seats) == dist_brute(seats), seats
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_two_pass_solution(seats: List[int]) -> int:
    n = len(seats)
    INF = n + 1
    left, right = [INF] * n, [INF] * n
    last = None
    for i in range(n):
        if seats[i]:
            last = i
        elif last is not None:
            left[i] = i - last
    last = None
    for i in range(n - 1, -1, -1):
        if seats[i]:
            last = i
        elif last is not None:
            right[i] = last - i
    return max(min(left[i], right[i]) for i in range(n) if seats[i] == 0)

assert exercise_4_two_pass_solution([1, 0, 0, 0, 1, 0, 1]) == 2
assert exercise_4_two_pass_solution([1, 0, 0, 0]) == 3
assert exercise_4_two_pass_solution([0, 1]) == 1
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 30)
    seats = [rng.randint(0, 1) for _ in range(n)]
    if sum(seats) in (0, n):
        continue
    assert exercise_4_two_pass_solution(seats) == dist_brute(seats), seats
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
from itertools import combinations

def exercise_5_k_people_solution(seats: List[int], k: int) -> int:
    n = len(seats)
    old = [j for j, c in enumerate(seats) if c]

    def feasible(D: int) -> bool:
        placed = list(old)                         # mọi người hiện có (cũ + mới), theo thứ tự thời gian đặt
        count = 0
        for i in range(n):
            if seats[i]:
                continue
            # cách mọi người đã đặt (cũ hoặc mới) ít nhất D, và cách mọi người cũ ít nhất D
            if all(abs(i - j) >= D for j in placed):
                placed.append(i)
                count += 1
                if count >= k:
                    return True
        return count >= k

    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo if feasible(lo) else 0

def k_people_oracle(seats, k):
    n = len(seats)
    empties = [i for i in range(n) if seats[i] == 0]
    old = [j for j in range(n) if seats[j]]
    best = 0
    for chosen in combinations(empties, k):
        new = list(chosen)
        q = min(min(abs(i - j) for j in old + [x for x in new if x != i]) for i in new)
        best = max(best, q)
    return best

assert exercise_5_k_people_solution([1, 0, 0, 0, 0, 0, 1], 1) == 3
assert exercise_5_k_people_solution([1, 0, 0, 0, 0, 0, 1], 2) == 2
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(3, 12)
    seats = [1 if rng.random() < 0.3 else 0 for _ in range(n)]
    if sum(seats) == 0:
        seats[rng.randrange(n)] = 1
    empties = n - sum(seats)
    if empties == 0:
        continue
    k = rng.randint(1, min(3, empties))
    assert exercise_5_k_people_solution(seats, k) == k_people_oracle(seats, k), (seats, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (một lượt)

```python
class Solution:
    def maxDistToClosest(self, seats: List[int]) -> int:
        first = last = None
        d = 0                                   # khoảng cách lớn nhất giữa hai người liên tiếp
        for i, c in enumerate(seats):
            if c:
                if last is not None:
                    d = max(d, i - last)
                if first is None:
                    first = i
                last = i
        return max(first, len(seats) - 1 - last, d // 2)
```

Công thức nhớ nhanh: **mép nguyên vẹn, giữa chia đôi**: `max(chỉ số người đầu, n - 1 - chỉ số người cuối, (khoảng cách lớn nhất hai người liên tiếp) // 2)`.

### Checklist trước khi submit

- [ ] Có xét **hai đầu** hàng ghế (`first` và `n - 1 - last`)?
- [ ] Hai đầu **không** chia đôi, khoảng giữa mới `// 2`?
- [ ] Khoảng giữa dùng **khoảng cách chỉ số** `b - a` (không phải số ghế trống)?
- [ ] Dùng `//` (số nguyên), không dùng `/`?
- [ ] `first` / `last` khởi tạo `None` để xử lý "chưa gặp người nào"?
- [ ] Đã thử: `[0, 1]`, `[1, 0]`, `[1, 0, 1]`, người chỉ ở một đầu, `n = 2 * 10^4`?
- [ ] Tên method đúng `maxDistToClosest(self, seats)` và trả `int`?